# 13.1 文本预处理与词法分析（2课时）

> **NOAI 竞赛课程 · 模块十三：自然语言处理**

---

## 本节目标

| 知识点 | 掌握程度 |
|--------|----------|
| NLP 基本概念与流程 | ⭐⭐⭐ |
| 分词方法（字/词/子词） | ⭐⭐⭐ |
| 中文分词 jieba | ⭐⭐⭐ |
| 英文分词 NLTK | ⭐⭐⭐ |
| 子词分词 BPE / WordPiece / SentencePiece | ⭐⭐ |
| 词汇表构建 | ⭐⭐⭐ |
| 文本清洗与规范化 | ⭐⭐⭐ |
| 词性标注 POS Tagging | ⭐⭐ |
| 命名实体识别 NER | ⭐⭐ |

---

## 一、NLP 概述

### 1.1 什么是自然语言处理？

**自然语言处理（Natural Language Processing, NLP）** 是人工智能的核心方向之一，旨在让计算机能够理解、生成和处理人类自然语言。

NLP 的主要任务层次：

```
┌─────────────────────────────────────────────┐
│              应用层                          │
│  机器翻译 / 问答系统 / 文本摘要 / 对话系统      │
├─────────────────────────────────────────────┤
│              任务层                          │
│  文本分类 / 情感分析 / NER / 关系抽取          │
├─────────────────────────────────────────────┤
│              表示层                          │
│  词嵌入 Word2Vec / GloVe / 句向量 BERT        │
├─────────────────────────────────────────────┤
│              基础层（本节重点）               │
│  分词 / 清洗 / POS / NER / 词汇表构建         │
└─────────────────────────────────────────────┘
```

### 1.2 NLP 基本流程

一个典型的 NLP 处理流水线：

```
原始文本 → 文本清洗 → 分词(Tokenization) → 词汇表构建(Vocabulary) → 数值化(Text → IDs) → 模型输入
```

**本节聚焦前四个步骤**，它们是一切 NLP 任务的基础。

---

## 二、分词（Tokenization）

### 2.1 为什么需要分词？

计算机无法直接处理原始文本字符串，需要将其拆分为最小的语义单位——**词元（Token）**。

- **中文**：词语之间没有空格，需要算法判断分词边界
- **英文**：以空格分隔，但需要处理缩写、标点等特殊情况

### 2.2 三种分词粒度

| 粒度 | 说明 | 示例 | 典型应用 |
|------|------|------|----------|
| **字级** | 每个字符单独切分 | "自然语言" → [自, 然, 语, 言] | 中文早期模型 |
| **词级** | 按完整单词切分 | "natural language" → [natural, language] | 传统NLP |
| **子词级** | 介于字符和词之间 | "unhappiness" → [un, happi, ness] | 现代预训练模型 |

### 2.3 中文分词 —— jieba 库

**jieba** 是 Python 中最流行的中文分词库，支持三种分词模式：

| 模式 | 说明 | 速度 |
|------|------|------|
| 精确模式 | 最精确，适合文本分析 | ⭐⭐ |
| 全模式 | 所有可能词的组合，有冗余 | ⭐⭐⭐ |
| 搜索引擎模式 | 精确模式基础上对长词再切分 | ⭐⭐ |

In [1]:
# 安装 jieba（如果尚未安装）
!pip install jieba -q

import jieba
print("jieba 导入成功！")

jieba 导入成功！


In [2]:
text = "我来到北京清华大学"

# 精确模式
seg_exact = jieba.cut(text, cut_all=False)
print("【精确模式】", "/".join(seg_exact))

# 全模式
seg_all = jieba.cut(text, cut_all=True)
print("【全模式】", "/".join(seg_all))

# 搜索引擎模式
seg_search = jieba.cut_for_search(text)
print("【搜索引擎模式】", "/".join(seg_search))

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\Terry\AppData\Local\Temp\jieba.cache
Loading model cost 0.460 seconds.
Prefix dict has been built successfully.


【精确模式】 我/来到/北京/清华大学
【全模式】 我/来到/北京/清华/清华大学/华大/大学
【搜索引擎模式】 我/来到/北京/清华/华大/大学/清华大学


In [3]:
# 自定义词典
text2 = "他来到了网易杭研大厦"

print("【添加自定义词典前】", "/".join(jieba.cut(text2)))

# 添加自定义词
jieba.add_word("杭研院")
print("【添加自定义词典后】", "/".join(jieba.cut(text2)))

【添加自定义词典前】 他/来到/了/网易/杭研/大厦
【添加自定义词典后】 他/来到/了/网易/杭研/大厦


In [4]:
# 字级分词示例（按字符切分）
text3 = "我来到北京清华大学，自然语言处理是有趣的！"
char_tokens = list(text3)
for i, ch in enumerate(char_tokens):
    print(f"No: {i}, Char: {ch}")

No: 0, Char: 我
No: 1, Char: 来
No: 2, Char: 到
No: 3, Char: 北
No: 4, Char: 京
No: 5, Char: 清
No: 6, Char: 华
No: 7, Char: 大
No: 8, Char: 学
No: 9, Char: ，
No: 10, Char: 自
No: 11, Char: 然
No: 12, Char: 语
No: 13, Char: 言
No: 14, Char: 处
No: 15, Char: 理
No: 16, Char: 是
No: 17, Char: 有
No: 18, Char: 趣
No: 19, Char: 的
No: 20, Char: ！


### 2.4 英文分词 —— NLTK

**NLTK（Natural Language Toolkit）** 是 Python 中经典的 NLP 工具库。

In [5]:
# 安装 NLTK
!pip install nltk -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print("nltk 导入成功！")

nltk 导入成功！


In [6]:
from nltk.tokenize import word_tokenize, sent_tokenize

en_text = "Natural Language Processing is fascinating! Isn't it amazing?"

# 按词分词
tokens = word_tokenize(en_text)
print("【原始文本】", en_text)
print("【word_tokenize】", tokens)

# 按句子分词
sentences = sent_tokenize(en_text)
print("【sent_tokenize】", sentences)

【原始文本】 Natural Language Processing is fascinating! Isn't it amazing?
【word_tokenize】 ['Natural', 'Language', 'Processing', 'is', 'fascinating', '!', 'Is', "n't", 'it', 'amazing', '?']
【sent_tokenize】 ['Natural Language Processing is fascinating!', "Isn't it amazing?"]


---

## 三、子词分词（Subword Tokenization）

### 3.1 为什么需要子词分词？

传统词级分词存在 **OOV（Out-of-Vocabulary）** 问题：
- 词汇表有限，遇到未登录词（如新词、专有名词）就无法处理
- 英文中词形变化多（run → running → ran），词表膨胀严重

子词分词的核心思想：
> **高频词保持完整，低频词拆分为更小的子词单元**

这样既不会产生过多 token，又能处理任何未知词。

### 3.2 BPE（Byte Pair Encoding）

BPE 是最经典的子词分词算法：

**算法步骤**：
1. 将所有词拆分为字符序列，并在词尾加 `</w>` 标记
2. 统计相邻字符对的出现频率
3. 合并频率最高的字符对为新符号
4. 重复步骤 2-3 直到达到预设的词表大小

**示例演示**：

```
初始:  {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w e s t </w>': 6}

第1轮: 最高频对 'e' + 's' → 'es'
  {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w es t </w>': 6}

第2轮: 最高频对 'es' + 't' → 'est'
  {'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est </w>': 6}

第3轮: 最高频对 'n' + 'e' → 'ne'
  {'l o w </w>': 5, 'l o w e r </w>': 2, 'ne w est </w>': 6}

...继续合并...
```

### 3.3 WordPiece

- BERT 模型使用的分词算法
- 与 BPE 类似，但选择合并对的标准不同：

$$\text{score}(A, B) = \frac{\text{freq}(A, B)}{\text{freq}(A) \times \text{freq}(B)}$$

- 使用 `##` 前缀标记非词首子词（如 `un` + `##happy` + `##ness`）

### 3.4 SentencePiece

- 直接在原始文本上训练，不需要预先分词
- 将文本视为 Unicode 字符序列
- 支持 BPE 和 Unigram 两种模型
- T5、XLNet 等模型采用此方法

In [7]:
# ===== BPE（Byte Pair Encoding）完整演示 =====

from collections import Counter, defaultdict

def get_pair_counts(word_splits):
    """统计相邻符号对频率"""
    pairs = Counter()
    for word, freq in word_splits.items():
        symbols = word
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(word_splits, pair):
    """合并指定字符对"""
    new_splits = {}
    merged = pair[0] + pair[1]
    for word, freq in word_splits.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_splits[tuple(new_word)] = freq
    return new_splits

# 初始化：语料词频
corpus = {
    'low': 5,
    'lower': 2,
    'newest': 6,
    'widest': 3
}

# 步骤1：拆分为字符序列，加 </w> 结束符
word_splits = {}
base_chars = set()
for word in corpus:
    chars = list(word) + ['</w>']
    word_splits[tuple(chars)] = corpus[word]
    base_chars.update(chars)

merges = []  # 记录合并历史

print("===== BPE 分词演示 =====")
print("初始化语料（字符级别）:")
for word_tuple, freq in word_splits.items():
    word_name = ''.join(w for w in word_tuple if w != '</w>')
    print(f"  {word_name:6s} → {list(word_tuple)}")

# 执行 5 轮合并
num_merges = 5
for step in range(num_merges):
    pairs = get_pair_counts(word_splits)
    if not pairs:
        break
    best_pair = pairs.most_common(1)[0][0]
    best_freq = pairs.most_common(1)[0][1]
    
    print(f"\n--- 合并第 {step+1} 轮 ---")
    print(f"最高频字符对: {best_pair} 出现 {best_freq} 次")
    print(f"合并后 → '{best_pair[0]}{best_pair[1]}'")
    
    word_splits = merge_pair(word_splits, best_pair)
    merges.append(best_pair)
    
    for word_tuple, freq in word_splits.items():
        word_name = ''.join(w for w in word_tuple if w != '</w>')
        print(f"  {word_name:6s} → {list(word_tuple)}")

# 展示最终词表
merged_symbols = [p[0] + p[1] for p in merges]
print("\n===== BPE 合并完成，最终词表 =====")
print(f"基础字符: {sorted(base_chars)}")
print(f"合并产生: {merged_symbols}")
print(f"完整词表大小: {len(base_chars) + len(merged_symbols)}")

===== BPE 分词演示 =====
初始化语料（字符级别）:
  low    → ['l', 'o', 'w', '</w>']
  lower  → ['l', 'o', 'w', 'e', 'r', '</w>']
  newest → ['n', 'e', 'w', 'e', 's', 't', '</w>']
  widest → ['w', 'i', 'd', 'e', 's', 't', '</w>']

--- 合并第 1 轮 ---
最高频字符对: ('e', 's') 出现 9 次
合并后 → 'es'
  low    → ['l', 'o', 'w', '</w>']
  lower  → ['l', 'o', 'w', 'e', 'r', '</w>']
  newest → ['n', 'e', 'w', 'es', 't', '</w>']
  widest → ['w', 'i', 'd', 'es', 't', '</w>']

--- 合并第 2 轮 ---
最高频字符对: ('es', 't') 出现 9 次
合并后 → 'est'
  low    → ['l', 'o', 'w', '</w>']
  lower  → ['l', 'o', 'w', 'e', 'r', '</w>']
  newest → ['n', 'e', 'w', 'est', '</w>']
  widest → ['w', 'i', 'd', 'est', '</w>']

--- 合并第 3 轮 ---
最高频字符对: ('est', '</w>') 出现 9 次
合并后 → 'est</w>'
  low    → ['l', 'o', 'w', '</w>']
  lower  → ['l', 'o', 'w', 'e', 'r', '</w>']
  newest</w> → ['n', 'e', 'w', 'est</w>']
  widest</w> → ['w', 'i', 'd', 'est</w>']

--- 合并第 4 轮 ---
最高频字符对: ('l', 'o') 出现 7 次
合并后 → 'lo'
  low    → ['lo', 'w', '</w>']
  lower  → ['lo', 'w', 'e',

---

## 四、词汇表构建（Vocabulary）

词汇表是将文本映射为数值的桥梁：

$$\text{token} \xleftrightarrow{\text{vocab}} \text{token\_id}$$

### 词汇表的关键元素

| 特殊 token | 含义 | 用途 |
|-----------|------|------|
| `<pad>` | 填充 | 对齐不同长度序列 |
| `<unk>` | 未知词 | 处理 OOV 词 |
| `<bos>` / `<eos>` | 句首/句尾 | 标记序列边界 |
| `<cls>` / `<sep>` | 分类/分隔 | BERT 风格标记 |

In [8]:
from collections import Counter

# ========== 构建词汇表 ==========

sentences = [
    "我 爱 自然 语言 处理",
    "我 喜欢 编程 代码",
    "你 喜欢 自然 语言",
    "我 爱 编程 代码"
]

# 特殊 token
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

# 收集所有 token 并统计频率
all_tokens = []
for sent in sentences:
    all_tokens.extend(sent.split())

token_counts = Counter(all_tokens)

# 构建词汇表：先放特殊token，再按频率排序
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, BOS_TOKEN: 2, EOS_TOKEN: 3}
idx = 4
for token, count in token_counts.most_common():
    vocab[token] = idx
    idx += 1

# 编码和解码函数
def encode(text, vocab):
    """文本 → token id 序列"""
    tokens = text.split()
    return [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens]

def decode(token_ids, id2token):
    """token id 序列 → 文本"""
    return [id2token.get(tid, UNK_TOKEN) for tid in token_ids]

# 演示
id2token = {v: k for k, v in vocab.items()}
test_text = "我 爱 自然 语言 处理"

encoded = encode(test_text, vocab)
decoded = decode(encoded, id2token)

print(f"原始句子: {test_text}")
print(f"编码结果: {encoded}")
print(f"解码结果: {decoded}")
print(f"\n完整词汇表:")
for token, tid in vocab.items():
    print(f"  {token}: {tid}")

原始句子: 我 爱 自然 语言 处理
编码结果: [4, 5, 6, 7, 11]
解码结果: ['我', '爱', '自然', '语言', '处理']

完整词汇表:
  <pad>: 0
  <unk>: 1
  <bos>: 2
  <eos>: 3
  我: 4
  爱: 5
  自然: 6
  语言: 7
  喜欢: 8
  编程: 9
  代码: 10
  处理: 11
  你: 12


---

## 五、文本清洗

### 5.1 常见清洗操作

```
原始文本 → 去除HTML标签 → 转小写 → 去除标点/特殊字符 → 去除多余空格 → 去除停用词 → 清洗后文本
```

| 操作 | 目的 | 示例 |
|------|------|------|
| 去除 HTML | 去除网页标签 | `<b>Hello</b>` → `Hello` |
| 转小写 | 统一大小写 | `Hello` → `hello` |
| 去标点 | 消除无意义符号 | `Hello!` → `Hello` |
| 去停用词 | 去除高频无义虚词 | `the, is, a` → 删除 |

In [9]:
import re
from nltk.corpus import stopwords

texts = [
    "The Quick Brown Fox jumps over the lazy dog!!!",
    "Natural Language Processing is REALLY interesting... isn't it?"
]

print("【原始文本】")
for t in texts:
    print(f"  {t}")

# 步骤1：去除标点
cleaned_punct = [re.sub(r'[^\w\s]', '', t) for t in texts]
print("\n【去除标点】")
for t in cleaned_punct:
    print(f"  {t}")

# 步骤2：转小写
cleaned_lower = [t.lower() for t in cleaned_punct]
print("\n【转小写】")
for t in cleaned_lower:
    print(f"  {t}")

# 步骤3：去停用词
en_stopwords = set(stopwords.words('english'))
result_no_stop = []
for t in cleaned_lower:
    tokens = t.split()
    filtered = [w for w in tokens if w not in en_stopwords]
    result_no_stop.append(filtered)

print("\n【去停用词后】")
for tokens in result_no_stop:
    print(f"  {tokens}")

【原始文本】
  The Quick Brown Fox jumps over the lazy dog!!!
  Natural Language Processing is REALLY interesting... isn't it?

【去除标点】
  The Quick Brown Fox jumps over the lazy dog
  Natural Language Processing is REALLY interesting isnt it

【转小写】
  the quick brown fox jumps over the lazy dog
  natural language processing is really interesting isnt it

【去停用词后】
  ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']
  ['natural', 'language', 'processing', 'really', 'interesting', 'isnt']


In [10]:
# ===== 中文文本清洗 =====

import re

zh_text = "我来到了北京清华大学，自然语言处理是有趣的！！！"

# 中文停用词列表（精简版）
zh_stopwords = {'的', '了', '是', '在', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好', '自己', '这'}

# 去除标点
cleaned = re.sub(r'[\s+\!\?\,\。\，\！\？\、\；\：\"\'\（\）\【\】]', '', zh_text)
print(f"【原始文本】 {zh_text}")
print(f"【去标点】 {cleaned}")

# jieba 分词 + 去停用词
import jieba
tokens = list(jieba.cut(cleaned))
filtered = [t for t in tokens if t not in zh_stopwords and len(t) > 0]
print(f"【去停用词后】 {filtered}")

【原始文本】 我来到了北京清华大学，自然语言处理是有趣的！！！
【去标点】 我来到了北京清华大学自然语言处理是有趣的
【去停用词后】 ['来到', '北京', '清华大学', '自然语言', '处理', '有趣']


---

## 六、文本规范化

### 6.1 词干提取（Stemming）

- 通过规则截取词的词干（不一定是一个合法单词）
- 速度快，但精度较低
- 常用算法：**Porter Stemmer**

示例：`running` → `runn`，`better` → `better`（不变）

### 6.2 词形还原（Lemmatization）

- 将词还原为词典中的原形（lemma）
- 需要词性信息，精度高
- 依赖词典（如 WordNet）

示例：`running` → `run`，`better` → `good`，`geese` → `goose`

In [11]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ['running', 'better', 'geese', 'corpora', 'studying', 'easily', 'went']

print(f"| {'原词':14s} | {'Stemming':12s} | {'Lemmatization':15s} |")
print(f"|{'-'*16}|{'-'*14}|{'-'*17}|")
for word in words:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos='v') if word == 'running' else lemmatizer.lemmatize(word)
    if word == 'went':
        lemma = lemmatizer.lemmatize(word, pos='v')
    print(f"| {word:14s} | {stem:12s} | {lemma:15s} |")

| 原词             | Stemming     | Lemmatization   |
|----------------|--------------|-----------------|
| running        | run          | run             |
| better         | better       | better          |
| geese          | gees         | goose           |
| corpora        | corpora      | corpus          |
| studying       | studi        | studying        |
| easily         | easili       | easily          |
| went           | went         | go              |


---

## 七、词性标注（POS Tagging）

词性标注是为每个词标注其语法类别（名词、动词、形容词等）。

### 常用词性标签（Penn Treebank）

| 标签 | 词性 | 示例 |
|------|------|------|
| NN | 名词（单数） | dog, cat |
| NNS | 名词（复数） | dogs, cats |
| VB | 动词（原形） | run, eat |
| VBD | 动词（过去式） | ran, ate |
| JJ | 形容词 | big, happy |
| RB | 副词 | quickly, very |

In [12]:
from nltk import pos_tag

sentence = "The quick brown fox jumps over the lazy dog"
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

tag_map = {
    'DT': '限定词', 'JJ': '形容词', 'NN': '名词', 'NNS': '名词(复数)',
    'VB': '动词', 'VBD': '动词(过去式)', 'VBZ': '动词-第三人称单数',
    'IN': '介词/连词', 'RB': '副词', 'PRP': '代词'
}

print(f"【句子】 {sentence}")
print(f"\n【词性标注结果】")
for word, tag in tagged:
    desc = tag_map.get(tag, tag)
    print(f"  {word:10s} → {tag:3s} ({desc})")

【句子】 The quick brown fox jumps over the lazy dog

【词性标注结果】
  The        → DT  (限定词)
  quick      → JJ  (形容词)
  brown      → NN  (名词)
  fox        → NN  (名词)
  jumps      → VBZ (动词-第三人称单数)
  over       → IN  (介词/连词)
  the        → DT  (限定词)
  lazy       → JJ  (形容词)
  dog        → NN  (名词)


---

## 八、命名实体识别（NER）

命名实体识别是从文本中识别并分类**命名实体**，如人名、地名、机构名等。

### BIO 标注方案

| 标签 | 含义 |
|------|------|
| B-PER | 人名开头 |
| I-PER | 人名内部 |
| B-ORG | 组织机构开头 |
| I-ORG | 组织机构内部 |
| B-LOC | 地名开头 |
| I-LOC | 地名内部 |
| O | 非实体 |

In [13]:
# NLTK 内置 NER
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

from nltk import ne_chunk

sentence = "Apple is looking at buying a U.K. startup for $1 billion in New York."
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

entities = ne_chunk(tagged)

print(f"【句子】 {sentence}")
print(f"\n【NER 结果】")

# 提取实体
for chunk in entities:
    if hasattr(chunk, 'label'):
        entity_text = ' '.join(c[0] for c in chunk)
        entity_type = chunk.label()
        type_map = {'ORG': '组织机构', 'GPE': '地缘政治实体', 'PERSON': '人名', 
                    'DATE': '日期', 'MONEY': '金额', 'FAC': '设施'}
        desc = type_map.get(entity_type, entity_type)
        print(f"  ('{entity_text}', '{entity_type}')  → {desc}")

【句子】 Apple is looking at buying a U.K. startup for $1 billion in New York.

【NER 结果】
  ('Apple', 'GPE')  → 地缘政治实体
  ('New York', 'GPE')  → 地缘政治实体


---

## 九、本节知识图谱

```
                        NLP 预处理流水线
                              │
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
         文本清洗        分词(Tokenization)  文本规范化
    ┌─────┬─────┐     ┌────┼────┐      ┌─────┼─────┐
    ▼     ▼     ▼     ▼    ▼    ▼      ▼     ▼     ▼
  去标点 转小写  字级 词级 子词(BPE)  Stemming Lemmatization POS
  去HTML 去空格  jieba NLTK WordPiece                       NER
  去停用词      SentencePiece
              │
              ▼
        词汇表构建(Vocabulary)
              │
              ▼
      token → token_id 编码
```

---

## 📝 练习题

### 练习 1：中文分词（基础）

使用 jieba 对以下句子进行精确模式分词，并统计各词的词频：

> "自然语言处理是人工智能的重要方向，它包括文本分类、机器翻译和情感分析等任务"

要求：
1. 使用 `jieba.cut` 进行精确分词
2. 使用 `collections.Counter` 统计词频
3. 打印每个词及其出现次数

---

### 练习 2：BPE 分词扩展（进阶）

修改本节 BPE 演示代码，使用以下新语料重新训练：

```python
corpus = {'higher': 5, 'lowest': 6, 'newer': 4, 'widest': 3}
```

执行 8 轮合并，观察最终词汇表。思考：为什么某些合并顺序如此？

---

### 练习 3：文本预处理流水线（综合）

编写一个完整的英文文本预处理函数 `preprocess(text)`，实现以下功能：

1. 转小写
2. 去除标点
3. NLTK 分词
4. 去除停用词
5. Porter Stemmer 词干提取

测试文本：
> "The students were studying Natural Language Processing in the classroom!"

---

### 练习 4：词汇表构建与应用（综合）

给定以下中文句子列表，构建完整词汇表并实现编码/解码：

```python
sentences = [
    "深度学习 改变了 自然语言处理",
    "预训练 模型 如 BERT 非常 强大",
    "Transformer 架构 是 现代 NLP 的 核心"
]
```

要求：
1. 构建包含 `<pad>` 和 `<unk>` 的词汇表
2. 实现 `encode()` 和 `decode()` 函数
3. 将所有句子编码为 ID 序列，并填充到等长